In [0]:
trips_df     = spark.table("pysparkdbt.silver.trips")
payments_df  = spark.table("pysparkdbt.silver.payments")
customers_df = spark.table("pysparkdbt.silver.customers")
drivers_df   = spark.table("pysparkdbt.silver.drivers")
vehicles_df  = spark.table("pysparkdbt.silver.vehicles")
locations_df = spark.table("pysparkdbt.silver.locations")

In [0]:
gold_base_df = trips_df.alias("t") \
    .join(payments_df.alias("p"), "trip_id", "left") \
    .join(customers_df.alias("c"), "customer_id", "left") \
    .join(drivers_df.alias("d"), "driver_id", "left") \
    .join(vehicles_df.alias("v"), "vehicle_id", "left")

In [0]:
from pyspark.sql.functions import *

gold_base_df = gold_base_df.select(
    col("t.trip_id"),
    col("t.customer_id"),
    col("t.driver_id"),
    col("t.vehicle_id"),
    col("t.trip_start_time"),
    col("t.trip_end_time"),
    col("t.distance_km"),
    col("p.amount")
)

In [0]:
gold_base_df = gold_base_df \
    .withColumn("trip_duration_min",
        (col("trip_end_time").cast("long") - col("trip_start_time").cast("long"))/60
    ) \
    .withColumn("trip_date", to_date("trip_start_time")) \
    .withColumn("trip_month", date_format("trip_start_time", "yyyy-MM"))

1. Total revenue per day

In [0]:
revenue_kpis = gold_base_df.groupBy("trip_date").agg(
    sum("amount").alias("total_revenue"),
    avg("amount").alias("avg_revenue"),
    count("trip_id").alias("total_trips")
)

display(revenue_kpis)

trip_date,total_revenue,avg_revenue,total_trips
2025-08-16,698.3699999999999,63.48818181818181,18
2025-09-11,1025.2999999999997,53.96315789473683,25
2025-07-20,395.06,49.3825,12
2025-07-24,411.47,68.57833333333333,14
2025-09-20,343.47999999999996,42.934999999999995,10
2025-08-06,425.39,38.67181818181818,14
2025-08-22,339.52,48.50285714285714,12
2025-08-25,451.1,75.18333333333334,10
2025-09-07,386.68999999999994,48.33624999999999,12
2025-09-17,490.19999999999993,54.46666666666666,12


2. Top customers based on spending

In [0]:
top_customers = gold_base_df.groupBy("customer_id").agg(
    sum("amount").alias("total_spent"),
    count("trip_id").alias("total_trips")
).orderBy(desc("total_spent")).limit(10)

top_customers = top_customers.join(
    customers_df.select("customer_id"),
    "customer_id",
    "left"
)

display(top_customers)

customer_id,total_spent,total_trips
159,797.7699999999999,17
198,792.86,15
194,788.54,14
44,715.56,14
80,699.4499999999999,14
16,682.31,13
18,674.9300000000001,12
10,632.25,10
112,613.5799999999999,10
42,595.55,10


Driver performance metrics

In [0]:
driver_kpis = gold_base_df.groupBy("driver_id").agg(
    count("trip_id").alias("total_trips"),
    avg("trip_duration_min").alias("avg_duration"),
    sum("distance_km").alias("total_distance")
)

display(driver_kpis)

driver_id,total_trips,avg_duration,total_distance
18,41,33.292682926829265,831.2000000000004
12,29,32.89655172413793,627.08
38,23,32.26086956521739,518.09
16,25,32.2,600.09
31,18,40.05555555555556,393.12999999999994
5,25,31.04,406.3899999999999
10,23,31.608695652173914,553.99
1,28,26.5,500.1700000000001
27,28,31.821428571428573,646.3299999999999
3,17,28.0,266.56000000000006


Vehicle utilization metrics

In [0]:
vehicle_kpis = gold_base_df.groupBy("vehicle_id").agg(
    count("trip_id").alias("total_trips"),
    sum("distance_km").alias("total_distance")
)

display(vehicle_kpis)

vehicle_id,total_trips,total_distance
18,41,831.2000000000004
12,29,627.08
38,23,518.09
16,25,600.09
31,18,393.12999999999994
5,25,406.3899999999999
10,23,553.99
1,28,500.1700000000001
27,28,646.3299999999999
3,17,266.56000000000006


Location Kpi

In [0]:
location_kpis = trips_df.groupBy("start_location").agg(
    count("trip_id").alias("trip_count")
).orderBy(desc("trip_count"))

display(location_kpis)

start_location,trip_count
Jenniferside,3
South Michael,3
South John,3
Christophermouth,2
East Corey,2
Jessicamouth,2
Lake Kyle,2
Adambury,2
East Lisa,2
Davishaven,2


Peak hour analysis

In [0]:
peak_hour_kpis = gold_base_df \
    .withColumn("hour", hour("trip_start_time")) \
    .groupBy("hour") \
    .agg(count("trip_id").alias("total_trips")) \
    .orderBy(desc("total_trips"))

display(peak_hour_kpis)

hour,total_trips
5,80
9,80
23,73
12,72
16,72
22,71
6,65
21,64
14,63
19,61


Monthly revenue trends

In [0]:
monthly_kpis = gold_base_df.groupBy("trip_month").agg(
    sum("amount").alias("monthly_revenue")
)

display(monthly_kpis)

trip_month,monthly_revenue
2025-06,6199.880000000001
2025-07,16578.440000000002
2025-09,10965.949999999995
2025-08,18083.550000000007


In [0]:
revenue_kpis.write.mode("overwrite").saveAsTable("pysparkdbt.gold.revenue_kpis")
top_customers.write.mode("overwrite").saveAsTable("pysparkdbt.gold.top_customers")
driver_kpis.write.mode("overwrite").saveAsTable("pysparkdbt.gold.driver_kpis")
vehicle_kpis.write.mode("overwrite").saveAsTable("pysparkdbt.gold.vehicle_kpis")
location_kpis.write.mode("overwrite").saveAsTable("pysparkdbt.gold.location_kpis")
peak_hour_kpis.write.mode("overwrite").saveAsTable("pysparkdbt.gold.peak_hour_kpis")
monthly_kpis.write.mode("overwrite").saveAsTable("pysparkdbt.gold.monthly_kpis")

In [0]:
gold_base_df.write.mode("overwrite").saveAsTable("pysparkdbt.gold.fact_trips")